# 技能1 · Day 4 上机：多模态融合与跨域对齐

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 实现多模态融合三策略（早融合/中融合/晚融合），理解各策略的适用场景与优劣
2. 从零实现对比学习损失（InfoNCE + CLIP对称损失），理解温度参数对分布的影响
3. 用 **transformers CLIP** 实现图文检索（产品图-文案相似度 + top-k检索）
4. 用 **transformers BLIP** 实现图文理解（自动描述生成 + VQA视觉问答）
5. 用 CLIP 实现零样本分类，理解跨域对齐的原理与营销应用
6. 设计企业级多模态架构（广告创意图文匹配系统），评估各模块延迟与瓶颈


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ CLIP/BLIP 模型首次运行需从 HuggingFace 下载：
> - `openai/clip-vit-base-patch32`（~600MB）
> - `Salesforce/blip-image-captioning-base`（~250MB）
> - `Salesforce/blip-vqa-base`（~250MB）
> 如网络受限，可设置 `HF_ENDPOINT=https://hf-mirror.com` 使用镜像。

In [ ]:
# !pip install transformers torch pillow numpy -q
# CLIP/BLIP模型首次运行需下载：
#   openai/clip-vit-base-patch32 (~600MB)
#   Salesforce/blip-image-captioning-base (~250MB)
#   Salesforce/blip-vqa-base (~250MB)

## 1. 数据集背景与营销映射

**营销场景**：多模态营销内容融合与对齐 -- 将产品图片、文案描述、结构化属性融合为统一表示。

| 模态 | 营销数据 | 编码模型 | 维度 |
|------|---------|---------|------|
| 文本 | 产品描述/广告文案 | sentence-transformers / CLIP文本编码器 | 384-512 |
| 图像 | 产品图片/广告创意 | CLIP-ViT / ResNet | 512-2048 |
| 结构化 | 价格/评分/库存 | MLP | 16-128 |

**核心任务**：
1. **融合策略**：如何将异构模态融合为统一表示？
2. **对齐**：如何让图文在共享空间中对齐（CLIP对比学习）？
3. **检索**：给定文案，如何检索最匹配的产品图片？
4. **理解**：如何让模型"看懂"产品图片（BLIP自动描述+VQA）？

> 💡 本上机使用 PIL 生成模拟产品图片，代码无需下载外部图片。实际项目中替换为真实产品图片即可。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import warnings
warnings.filterwarnings('ignore')

# 工具函数：生成模拟产品图片（无需下载外部图片）
def make_product_image(color, label="PRODUCT"):
    img = Image.new('RGB', (224, 224), color)
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 24)
    except:
        font = ImageFont.load_default()
    bbox = draw.textbbox((0, 0), label, font=font)
    x = (224 - (bbox[2] - bbox[0])) // 2
    y = (224 - (bbox[3] - bbox[1])) // 2
    draw.text((x, y), label, fill='white', font=font)
    return img

print("环境就绪。torch:", torch.__version__)

## 2. 多模态融合三策略

企业拥有多种数据（文本/图像/结构化）时，如何融合为统一表示？

| 策略 | 融合层 | 优势 | 劣势 | 营销场景 |
|------|--------|------|------|---------|
| 早融合 | 特征层 | 学习模态间交叉特征 | 要求模态同时可用 | 产品推荐（图文+价格+评分） |
| 中融合 | 注意力层 | 动态权重，自适应 | 计算量较大 | 跨模态注意力（文案关注图片区域） |
| 晚融合 | 决策层 | 模态解耦，可独立训练 | 无法捕捉交叉特征 | 多渠道归因（搜索/展示/社交） |

**关键公式**：
- 早融合：`z = MLP([z_text; z_image; z_struct])`
- 中融合（注意力）：`alpha_i = softmax(W * y_i), y = sum(alpha_i * y_i)`
- 晚融合：`y = w1*y_text + w2*y_image + w3*y_struct`

> 详见独立教材 3.4.1 节（多模态表示融合三种策略）

In [ ]:
# 1. 多模态融合三策略实现（早融合/中融合/晚融合）
# 营销场景：产品推荐 -- 融合产品文本描述、图片特征、结构化属性（价格/评分）

# 早融合：拼接所有模态特征 -> MLP
class EarlyFusion(nn.Module):
    def __init__(self, text_dim, image_dim, struct_dim, hidden_dim, output_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(text_dim + image_dim + struct_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, text_emb, image_emb, struct_emb):
        combined = torch.cat([text_emb, image_emb, struct_emb], dim=-1)
        return self.mlp(combined)

# 中融合：跨模态注意力（text query, image key/value）
class CrossModalAttentionFusion(nn.Module):
    def __init__(self, text_dim, image_dim, hidden_dim, output_dim, num_heads=4):
        super().__init__()
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.image_proj = nn.Linear(image_dim, hidden_dim)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=num_heads, batch_first=True)
        self.output_proj = nn.Linear(hidden_dim, output_dim)
    def forward(self, text_emb, image_emb):
        q = self.text_proj(text_emb).unsqueeze(1)  # (batch, 1, hidden)
        kv = self.image_proj(image_emb).unsqueeze(1)  # (batch, 1, hidden)
        attn_out, _ = self.attention(q, kv, kv)
        return self.output_proj(attn_out.squeeze(1))

# 晚融合：各模态独立预测 -> 学习权重加权
class LateFusion(nn.Module):
    def __init__(self, text_dim, image_dim, struct_dim, hidden_dim, output_dim):
        super().__init__()
        self.text_mlp = nn.Sequential(nn.Linear(text_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim))
        self.image_mlp = nn.Sequential(nn.Linear(image_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim))
        self.struct_mlp = nn.Sequential(nn.Linear(struct_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, output_dim))
        self.weights = nn.Parameter(torch.ones(3))  # 可学习融合权重
    def forward(self, text_emb, image_emb, struct_emb):
        y_text = self.text_mlp(text_emb)
        y_image = self.image_mlp(image_emb)
        y_struct = self.struct_mlp(struct_emb)
        w = torch.softmax(self.weights, dim=0)
        return w[0]*y_text + w[1]*y_image + w[2]*y_struct

# 测试
torch.manual_seed(42)
text_emb = torch.randn(4, 128)
image_emb = torch.randn(4, 256)
struct_emb = torch.randn(4, 16)

early = EarlyFusion(128, 256, 16, 128, 1)
mid = CrossModalAttentionFusion(128, 256, 128, 1)
late = LateFusion(128, 256, 16, 128, 1)

print("早融合输出:", early(text_emb, image_emb, struct_emb).shape)
print("中融合输出:", mid(text_emb, image_emb).shape)
print("晚融合输出:", late(text_emb, image_emb, struct_emb).shape)
print("晚融合权重:", torch.softmax(late.weights, dim=0).detach().tolist())

## 3. 对比学习（Contrastive Learning）

对比学习是 CLIP/SimCLR 等模型的核心技术：**通过拉近正样本对、推远负样本对来学习表示**。

**InfoNCE 损失**：
```
L = -log[ exp(sim(z, z+)/tau) / (exp(sim(z, z+)/tau) + sum(exp(sim(z, z-)/tau))) ]
```

- `z`：锚点（anchor），`z+`：正样本，`z-`：负样本
- `tau`：温度参数（tau小->分布尖锐，tau大->分布平坦）

**CLIP 的对称损失**：`L = (L_img2text + L_text2img) / 2`，双向对比。

**为什么对比学习有效**：不需要标签就能学到好的表示 -- 只要能定义"相似"和"不相似"。

**营销应用**：产品图片-文案对齐 -- 匹配的图文对为正样本，不匹配的为负样本。

> 详见独立教材 3.4.2 节（对比学习原理）

In [ ]:
# 2. 对比学习实现（InfoNCE loss + CLIP对称损失）
# 对比学习核心：拉近正样本对，推远负样本对

def info_nce_loss(anchor, positive, negatives, temperature=0.07):
    """InfoNCE损失：-log[exp(sim(z,z+)/tau) / (exp(sim(z,z+)/tau) + sum(exp(sim(z,z-)/tau)))]"""
    anchor = F.normalize(anchor, dim=-1)
    positive = F.normalize(positive, dim=-1)
    negatives = F.normalize(negatives, dim=-1)

    # 正样本相似度: (batch, 1)
    pos_sim = torch.sum(anchor * positive, dim=-1, keepdim=True) / temperature

    # 负样本相似度: (batch, num_neg)
    neg_sim = torch.bmm(negatives, anchor.unsqueeze(-1)).squeeze(-1) / temperature

    # 拼接logits: (batch, 1 + num_neg)，index 0 是正样本
    logits = torch.cat([pos_sim, neg_sim], dim=-1)
    labels = torch.zeros(anchor.size(0), dtype=torch.long)
    return F.cross_entropy(logits, labels)

def clip_loss(image_features, text_features, temperature=0.07):
    """CLIP对称对比损失：图文双向对比"""
    image_features = F.normalize(image_features, dim=-1)
    text_features = F.normalize(text_features, dim=-1)

    # 相似度矩阵: (batch, batch)
    logits = image_features @ text_features.T / temperature
    labels = torch.arange(image_features.size(0))

    # 对称损失
    loss_i2t = F.cross_entropy(logits, labels)       # 图->文
    loss_t2i = F.cross_entropy(logits.T, labels)     # 文->图
    return (loss_i2t + loss_t2i) / 2

# 测试
torch.manual_seed(42)
anchor = torch.randn(4, 128)
positive = anchor + 0.1 * torch.randn(4, 128)  # 正样本：与anchor相近
negatives = torch.randn(4, 10, 128)              # 10个负样本

print("=== InfoNCE 损失测试 ===")
for temp in [0.01, 0.07, 0.1, 1.0]:
    loss = info_nce_loss(anchor, positive, negatives, temperature=temp)
    print(f"  temperature={temp:.2f}: loss={loss.item():.4f}")

print("\n=== CLIP 对称损失测试 ===")
img_feats = torch.randn(4, 512)
txt_feats = torch.randn(4, 512)
for temp in [0.01, 0.07, 0.1]:
    loss = clip_loss(img_feats, txt_feats, temperature=temp)
    print(f"  temperature={temp:.2f}: loss={loss.item():.4f}")

# 验证：当正样本对齐时loss应更低
aligned_txt = img_feats + 0.01 * torch.randn(4, 512)
loss_aligned = clip_loss(img_feats, aligned_txt, temperature=0.07)
print(f"\n对齐后 loss={loss_aligned.item():.4f} (应比随机更低)")

## 4. CLIP：图文对齐的里程碑

CLIP（Contrastive Language-Image Pre-training，OpenAI 2021）用对比学习将图像和文本对齐到同一向量空间。

**核心架构**：双塔 -- 图像编码器（ViT）+ 文本编码器（Transformer），各自编码后投影到共享空间。

**关键 API**（transformers 库）：
- `CLIPModel.get_image_features()` -> 图像 embedding
- `CLIPModel.get_text_features()` -> 文本 embedding
- `CLIPModel(**inputs)` -> `outputs.logits_per_image`（相似度矩阵）

**从 CLIP 到 GPT-4o 的演进**：
| 阶段 | 模型 | 核心 |
|------|------|------|
| 对比学习对齐 | CLIP (2021) | 双塔+对比损失 |
| 视觉-语言预训练 | BLIP-2 (2023) | Q-Former桥接ViT+LLM |
| 原生多模态 | GPT-4o (2024) | 端到端统一token空间 |
| 开源多模态 | LLaVA (2024) | CLIP-ViT + 投影层 + LLM |

> 详见独立教材 3.2.3 节（从CLIP到GPT-4o的多模态演进）

In [ ]:
# 3. CLIP图文检索（transformers CLIPModel）
# 营销场景：用户搜索文案 -> 返回最匹配的产品图片

from transformers import CLIPProcessor, CLIPModel

# 加载CLIP模型（首次运行需下载~600MB）
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# 3个产品图片 + 营销文案
product_images = [
    make_product_image((180, 30, 30), "LIPSTICK"),    # 红色口红
    make_product_image((30, 120, 80), "SKINCARE"),    # 绿色护肤
    make_product_image((30, 50, 150), "GADGET"),      # 蓝色电子
]
product_texts = ["红色哑光口红彩妆", "绿色天然护肤面霜", "蓝色智能电子设备"]

# CLIP编码（图文同时输入）
inputs = clip_processor(text=product_texts, images=product_images, return_tensors="pt", padding=True)
outputs = clip_model(**inputs)

# 图文相似度矩阵
logits_per_image = outputs.logits_per_image  # (num_images, num_texts)
probs = logits_per_image.softmax(dim=-1)

print("=== 图文匹配结果 ===")
for i, txt in enumerate(product_texts):
    best_j = probs[i].argmax().item()
    print(f"图片{i}({txt}) -> 最佳匹配: {product_texts[best_j]} (p={probs[i][best_j]:.3f})")

# Top-k检索：给定文本query，检索最匹配的图片
query = "红色唇膏"
img_inputs = clip_processor(images=product_images, return_tensors="pt")
img_features = clip_model.get_image_features(**img_inputs)
img_features = img_features / img_features.norm(dim=-1, keepdim=True)

txt_inputs = clip_processor(text=[query], return_tensors="pt", padding=True)
txt_features = clip_model.get_text_features(**txt_inputs)
txt_features = txt_features / txt_features.norm(dim=-1, keepdim=True)

similarities = (img_features @ txt_features.T).squeeze()
ranked_indices = similarities.argsort(descending=True)

print(f"\n=== Top-k 检索: query='{query}' ===")
for rank, idx in enumerate(ranked_indices):
    print(f"  Rank {rank+1}: 图片{idx.item()} ({product_texts[idx]}) sim={similarities[idx]:.4f}")

## 5. BLIP-2：图文理解的突破

BLIP-2（Salesforce 2023）用 Q-Former 桥接冻结的视觉编码器和冻结的 LLM，实现图文理解。

**架构**：冻结ViT -> Q-Former（学习查询token）-> 冻结LLM

**与CLIP的区别**：
- CLIP 只做对齐（相似度），不生成文本
- BLIP-2 能生成描述、回答问题（生成式）

**API**（transformers库）：
- BLIP-2：`Blip2Processor` + `Blip2ForConditionalGeneration`（模型大，2.7B+）
- BLIP（轻量）：`BlipProcessor` + `BlipForConditionalGeneration`（~250MB）
- VQA：`BlipForQuestionAnswering`（视觉问答）

> 本上机用 BLIP-base 作为轻量替代。BLIP-2 API 几乎相同，仅类名和模型ID不同。

In [ ]:
# 4. BLIP图文理解（产品图自动描述 + VQA）
# 营销场景：自动为产品图片生成描述和回答产品相关问题
# 注：BLIP-2使用Blip2Processor/Blip2ForConditionalGeneration（API相似但模型更大2.7B+）
#     这里用BLIP-base作为轻量替代，首次运行需下载~250MB

from transformers import BlipProcessor, BlipForConditionalGeneration

# 加载BLIP模型
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# 生成产品图片
product_img = make_product_image((180, 30, 30), "LIPSTICK")

# (1) 自动描述生成（Image Captioning）
inputs = blip_processor(images=product_img, return_tensors="pt")
output_ids = blip_model.generate(**inputs, max_length=50, num_beams=5)
caption = blip_processor.decode(output_ids[0], skip_special_tokens=True)
print("=== 自动描述 ===")
print(f"产品图片描述: {caption}")

# (2) 条件描述生成（带prompt引导）
prompt = "a photo of"
inputs = blip_processor(images=product_img, text=prompt, return_tensors="pt")
output_ids = blip_model.generate(**inputs, max_length=50, num_beams=5)
caption_cond = blip_processor.decode(output_ids[0], skip_special_tokens=True)
print(f"条件描述 ('{prompt}'): {caption_cond}")

# (3) VQA -- 视觉问答（需要BlipForQuestionAnswering模型）
from transformers import BlipForQuestionAnswering

vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
vqa_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")

questions = [
    "What color is this product?",
    "What is this product?",
]

print("\n=== VQA 视觉问答 ===")
for q in questions:
    inputs = vqa_processor(images=product_img, text=q, return_tensors="pt")
    output_ids = vqa_model.generate(**inputs, max_length=10)
    answer = vqa_processor.decode(output_ids[0], skip_special_tokens=True)
    print(f"  Q: {q}")
    print(f"  A: {answer}")

## 6. 跨域对齐与零样本分类

CLIP 的对齐空间天然支持零样本分类：无需训练，只需将类别名转为文本，用CLIP计算图片与各类别的相似度。

**零样本分类流程**：
1. 定义类别：["护肤品", "彩妆", "食品", ...]
2. 转为prompt：["a photo of skincare", ...]
3. CLIP编码图片+文本
4. 计算相似度 -> softmax -> 取最大值

**营销应用**：新品上架自动分类、广告图片自动打标签。

**跨域对齐的本质**：CLIP将图像和文本映射到同一空间，使"图片向量与文本向量的余弦相似度"直接反映语义匹配度。

In [ ]:
# 5. 跨域对齐与零样本分类（CLIP zero-shot classification）
# 营销场景：新品上架图片自动分类打标签，无需训练

# 定义产品类别（营销场景）
categories = ["skincare product", "makeup cosmetic", "food snack", "electronic gadget", "clothing apparel"]
prompts = [f"a photo of {cat}" for cat in categories]

# 生成测试产品图片
test_images = [
    make_product_image((200, 180, 150), "CREAM"),     # 护肤品
    make_product_image((180, 30, 30), "LIPSTICK"),    # 彩妆
    make_product_image((200, 150, 50), "SNACK"),      # 食品
]

# CLIP零样本分类
inputs = clip_processor(text=prompts, images=test_images, return_tensors="pt", padding=True)
outputs = clip_model(**inputs)
logits = outputs.logits_per_image  # (num_images, num_categories)
probs = logits.softmax(dim=-1)

print("=== CLIP 零样本分类 ===")
for i in range(len(test_images)):
    print(f"\n图片{i}:")
    for j, cat in enumerate(categories):
        print(f"  {cat}: {probs[i][j]:.4f}")
    best = probs[i].argmax().item()
    print(f"  -> 预测: {categories[best]} (conf={probs[i][best]:.4f})")

# 跨域对齐分析：正确匹配 vs 错误匹配的相似度差距
print("\n=== 跨域对齐分析 ===")
img_inputs = clip_processor(images=test_images, return_tensors="pt")
img_features = clip_model.get_image_features(**img_inputs)
img_features = img_features / img_features.norm(dim=-1, keepdim=True)

txt_inputs = clip_processor(text=prompts, return_tensors="pt", padding=True)
txt_features = clip_model.get_text_features(**txt_inputs)
txt_features = txt_features / txt_features.norm(dim=-1, keepdim=True)

sim_matrix = img_features @ txt_features.T  # (3, 5)

correct_sims = []
wrong_sims = []
for i in range(len(test_images)):
    for j in range(len(categories)):
        sim = sim_matrix[i][j].item()
        if j == i:
            correct_sims.append(sim)
        else:
            wrong_sims.append(sim)

print(f"正确匹配相似度均值: {np.mean(correct_sims):.4f}")
print(f"错误匹配相似度均值: {np.mean(wrong_sims):.4f}")
print(f"对齐差距: {np.mean(correct_sims) - np.mean(wrong_sims):.4f} (越大越好)")

## 7. 企业级多模态架构设计

将融合策略、对比学习、CLIP/BLIP整合为企业级系统。

**架构设计原则**（独立教材 3.4.3 节）：
1. **分层解耦**：编码层、融合层、对齐层、存储层各自独立
2. **多存储共存**：向量数据库（语义检索）+ 图数据库（关系推理）
3. **端到端可训练**：从编码到对齐可通过对比学习端到端优化
4. **在线/离线分离**：产品embedding预计算（离线），用户embedding实时计算（在线）

**本任务目标**：设计一个广告创意图文匹配系统，画出架构图，评估各模块。

In [ ]:
# 6. 企业级多模态架构设计（广告创意图文匹配系统）
# 营销场景：广告投放前自动匹配文案与图片，提升CTR

architecture = """
======================================================================
              广告创意图文匹配系统架构
======================================================================

  数据层
  +----------+  +----------+  +----------+  +----------+
  | 广告文案  |  | 创意图片  |  | 投放数据  |  | 用户画像  |
  | (文本)   |  | (图像)   |  | (结构化)  |  | (行为)   |
  +----+-----+  +----+-----+  +----+-----+  +----+-----+
       |             |             |             |
  编码层
  +----v-----+  +----v-----+  +----v-----+  +----v-----+
  |sentence-  |  |CLIP-ViT  |  |  MLP     |  |Behavior  |
  |transformers| | (图像)   |  |(属性编码) |  |Encoder   |
  +----+-----+  +----+-----+  +----+-----+  +----+-----+
       |             |             |             |
  对齐层（CLIP对比学习）
  +----v-------------v-------------v-------------v-----+
  |         共享Embedding空间（512维）                  |
  |    对比学习对齐：匹配图文对拉近，不匹配推远           |
  +-----------------------+---------------------------+
                          |
  存储与检索层
  +-----------------------v---------------------------+
  |  FAISS IndexFlatIP (向量检索) + Redis (缓存)       |
  +-----------------------+---------------------------+
                          |
  应用层
  +----------+  +----------+  +----------+  +----------+
  | 图文匹配  |  | 创意搜索  |  | CTR预测   |  | 质量评分  |
  +----------+  +----------+  +----------+  +----------+
======================================================================
"""

print(architecture)

# 各模块定义与评估
modules = [
    {"name": "文本编码器", "model": "sentence-transformers/all-MiniLM-L6-v2",
     "dim": 384, "latency_ms": 5, "throughput": "2000/s"},
    {"name": "图像编码器", "model": "openai/clip-vit-base-patch32",
     "dim": 512, "latency_ms": 30, "throughput": "300/s"},
    {"name": "属性编码器", "model": "MLP(64->128)",
     "dim": 128, "latency_ms": 1, "throughput": "10000/s"},
    {"name": "对齐层", "model": "CLIP对比学习(projection 512)",
     "dim": 512, "latency_ms": 2, "throughput": "5000/s"},
    {"name": "向量检索", "model": "FAISS IndexFlatIP",
     "dim": 512, "latency_ms": 3, "throughput": "10000/s"},
]

print("=== 各模块评估 ===")
header = f"{'模块':<12} {'模型':<42} {'维度':<6} {'延迟(ms)':<10} {'吞吐':<10}"
print(header)
print("-" * 82)
for m in modules:
    print(f"{m['name']:<12} {m['model']:<42} {m['dim']:<6} {m['latency_ms']:<10} {m['throughput']:<10}")

# 系统延迟分析
total_latency = sum(m['latency_ms'] for m in modules)
# 并行编码：取编码层最大延迟 + 对齐 + 检索
parallel_latency = max(m['latency_ms'] for m in modules[:3]) + modules[3]['latency_ms'] + modules[4]['latency_ms']

print(f"\n=== 延迟分析 ===")
print(f"串行总延迟: {total_latency}ms")
print(f"并行编码延迟: {parallel_latency}ms (编码并行+对齐+检索)")
print(f"目标: <50ms {'达标' if parallel_latency < 50 else '未达标'}")

# 瓶颈分析
bottleneck = max(modules, key=lambda x: x['latency_ms'])
print(f"\n瓶颈: {bottleneck['name']} ({bottleneck['latency_ms']}ms)")
print(f"优化方向: 1) 用clip-vit-base-patch16(更小) 2) 批处理+GPU 3) 图像embedding预计算(离线)")

## 8. 反思与前沿

### 反思问题
1. 早融合 vs 晚融合：在什么场景下你会选哪个？（提示：模态是否总是同时可用？）
2. 温度参数 tau=0.01 vs tau=1.0，哪个更适合营销图文匹配？为什么？
3. CLIP零样本分类在什么情况下会失效？（提示：细粒度分类、领域偏移）
4. 从CLIP到GPT-4o，"原生多模态"解决了双塔架构的什么问题？

### 2026 前沿：原生多模态与对比学习
- **GPT-4o/Gemini**：端到端多模态训练，统一token空间处理文本/图像/音频
- **LLaVA**：开源视觉-语言模型（CLIP-ViT + 投影层 + LLM），低成本方案
- **对比学习仍是基础**：即使是原生多模态，预训练阶段仍大量使用对比学习
- **BLIP-3 / LLaVA-1.6**：2024-2025开源多模态持续进化

> 深入阅读见 reading.md 的 CLIP/BLIP-2/LLaVA 条目。